# OSM Pipeline — Orchestrator

Runs all three modules **in sequence** by executing each notebook programmatically.  
Source notebooks are **never modified** — each run saves an executed copy to `_pipeline_output/`.

```
osm_commercial_fetch.ipynb          → osm_commercial_NNN_*.csv
        ↓
osm_commercial_by_distances.ipynb   → osm_distances_NNN_*.csv
        ↓
osm_viz.ipynb                       → interactive maps & charts
```

---
### Team workflow
| Who | Edits | Does not touch |
|---|---|---|
| Your teammate | `osm_commercial_fetch.ipynb` | the other two |
| You | `osm_commercial_by_distances.ipynb`, `osm_viz.ipynb` | fetch notebook |
| Anyone | `osm_pipeline.ipynb` (this file) | — |

To run the full pipeline: **Restart kernel → Run All Cells**  
To re-run only one module: set the others to `False` in **Cell 2**.

Kernel: **OSMnx Scraper (Python 3.11)**

## Cell 1 · Imports

In [1]:
import papermill as pm
import pathlib, sys, time
from datetime import datetime

# Output folder for executed notebook copies (source notebooks are never modified)
OUTPUT_DIR = pathlib.Path("_pipeline_output")
OUTPUT_DIR.mkdir(exist_ok=True)

RUN_ID  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"papermill {pm.__version__}")
print(f"Run ID   : {RUN_ID}")
print(f"Outputs  : {OUTPUT_DIR}/")


papermill 2.7.0
Run ID   : 2026-04-30_17-50-18
Outputs  : _pipeline_output/


## Cell 2 · Configuration

Set any module to `False` in `RUN_MODULE` to skip it on this run.  
Useful when you only want to re-run visualization without re-fetching data.

In [2]:
# ── PIPELINE CONFIGURATION ────────────────────────────────────────────────────
# List of notebooks to execute in order.
# Do NOT change the order — each module depends on the output of the previous one.

PIPELINE = [
    ("osm_commercial_fetch.ipynb",                              "Module 1 · Fetch commercial features"),
    ("osm_commercial_by_distances.ipynb", "Module 2 · Add proximity distances"),
    ("osm_viz.ipynb",                     "Module 3 · Generate visualizations"),
]

# Set to False to skip a module (useful for partial re-runs)
RUN_MODULE = {
    "osm_commercial_fetch.ipynb":                               True,
    "osm_distances_to_POIs/osm_commercial_by_distances.ipynb":  True,
    "osm_distances_to_POIs/osm_viz.ipynb":                      True,
}

print("Pipeline:")
for nb, desc in PIPELINE:
    status = "RUN" if RUN_MODULE.get(nb, True) else "SKIP"
    print(f"  [{status}]  {desc}")


Pipeline:
  [RUN]  Module 1 · Fetch commercial features
  [RUN]  Module 2 · Add proximity distances
  [RUN]  Module 3 · Generate visualizations


## Cell 3 · Run Pipeline

Executes each enabled notebook in order.  
If a module fails, the pipeline stops and shows the error — fix it and re-run.

In [3]:
results = []

for nb_path, description in PIPELINE:
    if not RUN_MODULE.get(nb_path, True):
        print(f"  SKIP  {description}")
        results.append((nb_path, "skipped", 0))
        continue

    if not pathlib.Path(nb_path).exists():
        raise FileNotFoundError(f"Notebook not found: {nb_path}")

    # Use only the filename (no subfolder) for the output copy name
    nb_stem = pathlib.Path(nb_path).stem
    out_path = OUTPUT_DIR / f"{nb_stem}_{RUN_ID}.ipynb"

    print(f"\n{'='*55}")
    print(f"  {description}")
    print(f"{'='*55}")
    print(f"  Source : {nb_path}")
    print(f"  Output : {out_path}")

    t0 = time.time()
    try:
        pm.execute_notebook(
            nb_path,
            str(out_path),
            kernel_name="osmnx-scraper",
            progress_bar=False,
        )
        elapsed = time.time() - t0
        print(f"  Status : OK  ({elapsed:.1f} s)")
        results.append((nb_path, "ok", elapsed))

    except pm.exceptions.PapermillExecutionError as e:
        elapsed = time.time() - t0
        print(f"  Status : FAILED  ({elapsed:.1f} s)")
        print(f"  Error  : {e}")
        results.append((nb_path, "failed", elapsed))
        print("\nPipeline stopped — fix the error above and re-run.")
        break

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("  PIPELINE SUMMARY")
print(f"{'='*55}")
for nb_path, status, elapsed in results:
    icon = {"ok": "OK", "failed": "FAIL", "skipped": "SKIP"}.get(status, "?")
    t_str = f"{elapsed:.1f} s" if elapsed else "-"
    print(f"  [{icon}]  {nb_path:<55} {t_str}")
print(f"{'='*55}")



  Module 1 · Fetch commercial features
  Source : osm_commercial_fetch.ipynb
  Output : _pipeline_output\osm_commercial_fetch_2026-04-30_17-50-18.ipynb


e:\IAAC Local GIT Repositories\OSMnx-data-scraper\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


  Status : OK  (5.1 s)

  Module 2 · Add proximity distances
  Source : osm_commercial_by_distances.ipynb
  Output : _pipeline_output\osm_commercial_by_distances_2026-04-30_17-50-18.ipynb
  Status : OK  (70.9 s)

  Module 3 · Generate visualizations
  Source : osm_viz.ipynb
  Output : _pipeline_output\osm_viz_2026-04-30_17-50-18.ipynb
  Status : OK  (12.0 s)

  PIPELINE SUMMARY
  [OK]  osm_commercial_fetch.ipynb                              5.1 s
  [OK]  osm_commercial_by_distances.ipynb                       70.9 s
  [OK]  osm_viz.ipynb                                           12.0 s
